<a href="https://colab.research.google.com/github/cybertiagovieira/US-CTI-Regulatory-Tracker/blob/main/federal_register_pull.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests
import json
from datetime import datetime

# Federal Register Agency Slugs mapped to your CTI list
AGENCY_SLUGS = {
    "OCC": "comptroller-of-the-currency",
    "FDIC": "federal-deposit-insurance-corporation",
    "FRB": "federal-reserve-system",
    "SEC": "securities-and-exchange-commission",
    "CFTC": "commodity-futures-trading-commission",
    "FinCEN": "financial-crimes-enforcement-network",
    "CFPB": "consumer-financial-protection-bureau",
    "OFAC": "foreign-assets-control-office",
    "Treasury": "treasury-department",
    "NCUA": "national-credit-union-administration",
    "FCA": "farm-credit-administration",
    "FHFA": "federal-housing-finance-agency",
    "IRS": "internal-revenue-service"
}

def fetch_fr_data(start_date, end_date):
    base_url = "https://www.federalregister.gov/api/v1/articles.json"

    # Construct conditions array for URL
    agency_params = [f"conditions[agencies][]={slug}" for slug in AGENCY_SLUGS.values()]
    agency_query = "&".join(agency_params)

    # Restrict to Rules and Proposed Rules for high-fidelity changes
    type_query = "conditions[type][]=RULE&conditions[type][]=PRORULE"
    date_query = f"conditions[publication_date][gte]={start_date}&conditions[publication_date][lte]={end_date}"

    full_url = f"{base_url}?{agency_query}&{type_query}&{date_query}&per_page=1000"

    response = requests.get(full_url)
    if response.status_code != 200:
        print(f"API Error: {response.status_code}")
        return []

    data = response.json()
    results = data.get('results', [])

    dashboard_payload = []

    for item in results:
        # Reverse map FR agency slug back to CTI Acronym
        item_agencies = [agency['raw_name'] for agency in item.get('agencies', [])]
        matched_agency = "Multiple/Treasury" # Default fallback
        for acronym, slug in AGENCY_SLUGS.items():
            if any(slug.replace("-", " ") in raw.lower() for raw in item_agencies):
                matched_agency = acronym
                break

        # Map FR Type to Dashboard Type
        doc_type = "Final Rule" if item.get('type') == "Rule" else "NPRM"

        entry = {
            "date": item.get('publication_date'),
            "agency": matched_agency,
            "type": doc_type,
            "title": item.get('title'),
            "summary": item.get('abstract', 'No summary provided by agency.')
        }
        dashboard_payload.append(entry)

    return dashboard_payload

# Execution Example for January 2026
if __name__ == "__main__":
    raw_intelligence = fetch_fr_data("2026-01-01", "2026-01-31")
    print(json.dumps(raw_intelligence, indent=2))